# Alltoall — MoE Expert Routing

## What alltoall does

Alltoall is a **distributed transpose**. Every rank sends a distinct chunk to every other
rank, and receives a distinct chunk from every other rank.

```
Before (rank i holds [data_for_rank_0, data_for_rank_1, data_for_rank_2, data_for_rank_3]):
  Rank 0: [A0, A1, A2, A3]
  Rank 1: [B0, B1, B2, B3]
  Rank 2: [C0, C1, C2, C3]
  Rank 3: [D0, D1, D2, D3]

After alltoall (rank i receives column i from everyone):
  Rank 0: [A0, B0, C0, D0]   ← all data destined for rank 0
  Rank 1: [A1, B1, C1, D1]
  Rank 2: [A2, B2, C2, D2]
  Rank 3: [A3, B3, C3, D3]
```

**alltoallv** is the variable-length variant: each rank can send and receive *different*
amounts to/from each peer. This is what MoE routing actually needs.

## Why MoE needs alltoall

In Mixture-of-Experts (MoE), a router decides which expert should process each token.
Experts are distributed across GPUs — GPU 0 owns experts 0..K, GPU 1 owns experts K+1..2K,
etc. After routing, token i needs to be on whichever GPU owns the selected expert.

```
Token routing (example, 8 tokens, 4 GPUs, 8 experts):

  Tok 0 → Expert 5 (on GPU 1)     GPU 0 needs to send tok 0 to GPU 1
  Tok 1 → Expert 2 (on GPU 0)     GPU 0 keeps tok 1
  Tok 2 → Expert 7 (on GPU 3)     GPU 0 needs to send tok 2 to GPU 3
  Tok 3 → Expert 1 (on GPU 0)     GPU 0 keeps tok 3
  ...

The routing is irregular — different numbers of tokens go to each GPU.
That irregularity is why we need alltoallv (variable counts), not alltoall.
```

After expert computation, the tokens must be routed back to their origin GPUs
(a second alltoall, transposing back).

## The Full MoE Communication Pattern

```
1. Router scores tokens:     [local compute, no communication]
2. alltoallv (dispatch):     tokens → expert GPUs
3. Expert FFN compute:       [local compute on each GPU]
4. alltoallv (combine):      expert outputs → origin GPUs
5. Weighted combine:         [local compute, combine expert outputs per token]
```

Steps 2 and 4 are the two alltoallv calls per MoE layer. They are latency-critical.

:::{note}
All cells use the **launcher pattern**: MPI workload is written to a temp script and
executed via `mpirun` as a subprocess.
:::

## 0. Environment Check

In [ ]:
import subprocess, os

def check(label, cmd, expect_in=None):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    ok = result.returncode == 0 and (expect_in is None or expect_in in result.stdout + result.stderr)
    print(f"{'OK' if ok else 'FAIL'}  {label}")
    if not ok:
        print(f"     {result.stdout.strip()[:200] or result.stderr.strip()[:200]}")
    return ok

check("mpirun available",          "which mpirun")
check("Intel MPI loaded",          "mpirun --version", expect_in="Intel")
check("PyTorch installed",         "python -c 'import torch; print(torch.__version__)'")
check("oneCCL bindings installed", "python -c 'import oneccl_bindings_for_pytorch'")
check("XPU available",             "python -c 'import torch; assert torch.xpu.is_available()'")

## 1. Basic Alltoall — Uniform Count, Correctness Verification

Start with the fixed-count variant to verify correctness before adding the routing
irregularity.

Rank i sends `count` elements to each peer. After alltoall, rank i holds `count` elements
from each peer. The result should be a matrix where every row is filled with the sender's
rank ID.

In [ ]:
%%writefile /tmp/ccl_alltoall_basic.py
import os
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

os.environ.setdefault("CCL_ATL_TRANSPORT", "ofi")
os.environ.setdefault("CCL_WORKER_COUNT", "1")
os.environ.setdefault("CCL_LOG_LEVEL", "warn")

dist.init_process_group(backend="ccl")

rank       = dist.get_rank()
world_size = dist.get_world_size()
device     = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

# Simulate: each rank has `tokens_per_expert` tokens for each expert GPU
# In uniform case: every GPU sends the same number of tokens to every other GPU
tokens_per_expert = 4
hidden_dim        = 512
chunk_size        = tokens_per_expert * hidden_dim

# send_buf[j*chunk_size : (j+1)*chunk_size] is what we send to rank j
# Fill with sender rank ID so we can verify who sent what
send_buf = torch.full(
    (world_size * chunk_size,),
    fill_value=float(rank),
    dtype=torch.bfloat16,
    device=device
)
recv_buf = torch.zeros_like(send_buf)

if rank == 0:
    print(f"tokens_per_expert={tokens_per_expert}, hidden={hidden_dim}")
    print(f"send_buf shape: {tuple(send_buf.shape)}  ({send_buf.numel()*2/1024:.1f} KB)")

# dist.all_to_all_single: fixed-count alltoall
dist.all_to_all_single(recv_buf, send_buf)
torch.xpu.synchronize(device)

# Verify: recv_buf[j*chunk_size:(j+1)*chunk_size] should be filled with j
errors = 0
for sender in range(world_size):
    chunk = recv_buf[sender*chunk_size : (sender+1)*chunk_size]
    expected = float(sender)
    if abs(chunk[0].item() - expected) > 0.01:
        errors += 1
        print(f"[rank {rank}] chunk from sender {sender}: got {chunk[0].item()}, expected {expected}")

status = "PASS" if errors == 0 else f"FAIL ({errors} errors)"
print(f"[rank {rank}] alltoall correctness: {status}")
if rank == 0:
    print("recv_buf[:4] values (expect [0,0,0,0] = from rank 0):")
    print(f"  {recv_buf[:4].tolist()}")
    print(f"recv_buf[chunk_size:chunk_size+4] values (expect [1,1,1,1] = from rank 1):")
    print(f"  {recv_buf[chunk_size:chunk_size+4].tolist()}")

dist.destroy_process_group()

In [ ]:
import subprocess
result = subprocess.run(
    "mpirun -n 4 -ppn 4 python /tmp/ccl_alltoall_basic.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

## 2. Alltoallv — Variable Token Counts (Real MoE Routing)

In practice, routing is imbalanced. Token 0 might route to expert 2 (GPU 0), token 1 to
expert 7 (GPU 1), etc. The number of tokens each GPU sends to each peer is unknown until
the routing decision is made at runtime.

`dist.all_to_all_single` supports variable counts via `output_split_sizes` and
`input_split_sizes` (the number of elements to send/receive to/from each peer).

**Critical invariant:** `my input_split_sizes[j]` must equal peer j's `output_split_sizes[my_rank]`.
If this is violated, you get silently wrong data or a hang.

In [ ]:
%%writefile /tmp/ccl_alltoallv_moe.py
import os
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

os.environ.setdefault("CCL_ATL_TRANSPORT", "ofi")
os.environ.setdefault("CCL_WORKER_COUNT", "1")
os.environ.setdefault("CCL_LOG_LEVEL", "warn")

dist.init_process_group(backend="ccl")

rank       = dist.get_rank()
world_size = dist.get_world_size()
device     = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

hidden_dim = 512

# Simulate imbalanced routing: each rank decides how many tokens go to each peer
# In a real system, this comes from the router's top-k selection
# Here we use a fixed pattern for reproducibility:
#   rank 0 sends: [3 to r0, 1 to r1, 2 to r2, 0 to r3]
#   rank 1 sends: [2 to r0, 4 to r1, 0 to r2, 1 to r3]
#   rank 2 sends: [1 to r0, 0 to r1, 3 to r2, 2 to r3]
#   rank 3 sends: [0 to r0, 2 to r1, 1 to r2, 3 to r3]

# token_counts_to_send[rank][dst] = number of tokens rank sends to dst
token_counts_to_send = [
    [3, 1, 2, 0],  # rank 0's dispatch
    [2, 4, 0, 1],  # rank 1's dispatch
    [1, 0, 3, 2],  # rank 2's dispatch
    [0, 2, 1, 3],  # rank 3's dispatch
]

# My send counts (tokens I send to each peer, each token = hidden_dim elements)
my_send_token_counts = token_counts_to_send[rank]  # [n_to_r0, n_to_r1, n_to_r2, n_to_r3]
input_split_sizes    = [c * hidden_dim for c in my_send_token_counts]

# My receive counts (tokens I receive from each peer)
# rank j sends token_counts_to_send[j][rank] tokens to me
my_recv_token_counts = [token_counts_to_send[j][rank] for j in range(world_size)]
output_split_sizes   = [c * hidden_dim for c in my_recv_token_counts]

total_send = sum(my_send_token_counts)
total_recv = sum(my_recv_token_counts)

# Build send buffer: tokens filled with sender rank ID for verification
send_buf = torch.full(
    (total_send * hidden_dim,),
    fill_value=float(rank),
    dtype=torch.bfloat16,
    device=device
)
recv_buf = torch.zeros(total_recv * hidden_dim, dtype=torch.bfloat16, device=device)

if rank == 0:
    print(f"Routing table (tokens sent per rank to each expert GPU):")
    for r, counts in enumerate(token_counts_to_send):
        print(f"  Rank {r}: {counts} → total dispatched={sum(counts)}")
    print()

dist.barrier()
print(f"[rank {rank}] sending {total_send} tokens, expecting {total_recv} tokens back")
print(f"[rank {rank}] input_split_sizes (elements): {input_split_sizes}")
print(f"[rank {rank}] output_split_sizes (elements): {output_split_sizes}")
dist.barrier()

# Alltoallv dispatch
dist.all_to_all_single(
    recv_buf, send_buf,
    output_split_sizes=output_split_sizes,
    input_split_sizes=input_split_sizes
)
torch.xpu.synchronize(device)

# Verify: each received chunk should have the value of the sender's rank
errors = 0
offset = 0
for sender in range(world_size):
    n_recv = my_recv_token_counts[sender]
    if n_recv == 0:
        continue
    chunk = recv_buf[offset : offset + n_recv * hidden_dim]
    expected_val = float(sender)
    if abs(chunk[0].item() - expected_val) > 0.01:
        errors += 1
        print(f"[rank {rank}] from sender {sender}: got {chunk[0].item():.1f}, expected {expected_val:.1f}")
    offset += n_recv * hidden_dim

dist.barrier()
status = "PASS" if errors == 0 else f"FAIL ({errors} errors)"
print(f"[rank {rank}] alltoallv correctness: {status}")

dist.destroy_process_group()

In [ ]:
result = subprocess.run(
    "mpirun -n 4 -ppn 4 python /tmp/ccl_alltoallv_moe.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

## 3. Full MoE Round-Trip: Dispatch → Expert Compute → Combine

This simulates the complete MoE communication pattern for one layer:
1. Alltoallv dispatch (tokens to experts)
2. Local expert FFN (simulated as a linear)
3. Alltoallv combine (expert outputs back to origin ranks)

In [ ]:
%%writefile /tmp/ccl_moe_roundtrip.py
import os, time
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

os.environ["CCL_ATL_TRANSPORT"] = "ofi"
os.environ["CCL_WORKER_COUNT"]  = "1"
os.environ["CCL_LOG_LEVEL"]     = "error"

dist.init_process_group(backend="ccl")
rank       = dist.get_rank()
world_size = dist.get_world_size()
device     = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

# Config: Mixtral-8x7B style MoE, top-2 routing
total_tokens  = 64       # tokens this rank holds (batch × seq_shard)
hidden_dim    = 4096
experts_per_gpu = 2      # number of experts owned by each GPU
top_k         = 2        # each token routes to 2 experts

torch.manual_seed(rank)

# Simulate router: randomly assign each token to `top_k` experts
# In reality: softmax over expert scores, then top-k selection
total_experts  = world_size * experts_per_gpu
# token_expert_assignment[t] = list of expert IDs for token t
selected_experts = torch.randint(0, total_experts, (total_tokens, top_k), device=device)

# Compute dispatch counts: how many (token, copy) tuples go to each GPU
# expert e lives on GPU e // experts_per_gpu
send_counts = [0] * world_size  # element count (not token count) to each peer
for t in range(total_tokens):
    for e in selected_experts[t].tolist():
        gpu_id = e // experts_per_gpu
        send_counts[gpu_id] += hidden_dim  # one token-hidden vector per (token, expert) pair

# Exchange send counts so each rank knows how much to receive
send_counts_t = torch.tensor(send_counts, dtype=torch.int64, device=device)
recv_counts_t = torch.zeros(world_size, dtype=torch.int64, device=device)
dist.all_to_all_single(recv_counts_t, send_counts_t)
recv_counts = recv_counts_t.tolist()

total_send = sum(send_counts)
total_recv = sum(recv_counts)

# Build dispatch buffer (in practice, gather token vectors by selected experts)
dispatch_buf = torch.randn(total_send, dtype=torch.bfloat16, device=device)
recv_buf     = torch.empty(total_recv, dtype=torch.bfloat16, device=device)

WARMUP = 3
ITERS  = 20

for _ in range(WARMUP):
    dist.all_to_all_single(recv_buf, dispatch_buf,
                           output_split_sizes=recv_counts,
                           input_split_sizes=send_counts)
    torch.xpu.synchronize(device)
dist.barrier()

dispatch_times = []
compute_times  = []
combine_times  = []

for _ in range(ITERS):
    # === Dispatch alltoallv ===
    t0 = time.perf_counter()
    dist.all_to_all_single(recv_buf, dispatch_buf,
                           output_split_sizes=recv_counts,
                           input_split_sizes=send_counts)
    torch.xpu.synchronize(device)
    t1 = time.perf_counter()
    dispatch_times.append((t1 - t0) * 1e3)

    # === Expert FFN compute (simulated) ===
    # In reality: each GPU applies its 2 expert FFN networks to received tokens
    expert_output = recv_buf * 1.0 + 0.01  # placeholder: scale + bias
    torch.xpu.synchronize(device)
    t2 = time.perf_counter()
    compute_times.append((t2 - t1) * 1e3)

    # === Combine alltoallv (reverse the transpose) ===
    combined_buf = torch.empty(total_send, dtype=torch.bfloat16, device=device)
    dist.all_to_all_single(combined_buf, expert_output,
                           output_split_sizes=send_counts,   # transposed
                           input_split_sizes=recv_counts)    # transposed
    torch.xpu.synchronize(device)
    t3 = time.perf_counter()
    combine_times.append((t3 - t2) * 1e3)

dist.barrier()

if rank == 0:
    def p50(lst): return sorted(lst)[len(lst)//2]
    dispatch_bytes = sum(send_counts) * 2  # BF16
    print(f"MoE Round-Trip Benchmark ({total_tokens} tokens, hidden={hidden_dim}, top_k={top_k})")
    print(f"Send counts to peers: {send_counts}")
    print(f"Recv counts from peers: {recv_counts}")
    print()
    print(f"  Dispatch alltoallv (p50): {p50(dispatch_times):.2f} ms")
    print(f"  Expert compute     (p50): {p50(compute_times):.2f} ms  [simulated]")
    print(f"  Combine  alltoallv (p50): {p50(combine_times):.2f} ms")
    print(f"  Total MoE comm     (p50): {p50(dispatch_times)+p50(combine_times):.2f} ms")

dist.destroy_process_group()

In [ ]:
result = subprocess.run(
    "mpirun -n 4 -ppn 4 python /tmp/ccl_moe_roundtrip.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

## 4. Load Imbalance: Why It Matters and How to Measure It

MoE alltoallv is a collective — all ranks must wait for the slowest rank to finish sending
before anyone can proceed. If one rank dispatches 3× more tokens than another, every rank
waits for that slow rank.

**Expert load imbalance** is one of the main sources of MoE latency variance in production.

In [ ]:
%%writefile /tmp/ccl_moe_imbalance.py
import os, time
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

os.environ["CCL_ATL_TRANSPORT"] = "ofi"
os.environ["CCL_WORKER_COUNT"]  = "1"
os.environ["CCL_LOG_LEVEL"]     = "error"

dist.init_process_group(backend="ccl")
rank       = dist.get_rank()
world_size = dist.get_world_size()
device     = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

hidden_dim = 4096
ITERS      = 30

# Compare balanced vs. imbalanced routing
scenarios = [
    # (label, send_counts_per_rank_scenario)
    # Each list entry is [tokens_to_r0, tokens_to_r1, tokens_to_r2, tokens_to_r3]
    ("balanced (8 each)",   [[8, 8, 8, 8]] * world_size),
    ("mild imbalance 2:1",  [[4, 4, 16, 8], [8, 8, 8, 8], [8, 8, 4, 4], [12, 8, 4, 8]]),
    ("severe imbalance 4:1",[[2, 2, 2, 26], [8, 8, 8, 8], [8, 8, 8, 8], [16, 8, 4, 4]]),
]

for label, all_rank_counts in scenarios:
    my_token_counts = all_rank_counts[rank]
    send_counts = [c * hidden_dim for c in my_token_counts]

    send_counts_t = torch.tensor(send_counts, dtype=torch.int64, device=device)
    recv_counts_t = torch.zeros(world_size, dtype=torch.int64, device=device)
    dist.all_to_all_single(recv_counts_t, send_counts_t)
    recv_counts = recv_counts_t.tolist()

    send_buf = torch.randn(sum(send_counts), dtype=torch.bfloat16, device=device)
    recv_buf = torch.empty(sum(recv_counts), dtype=torch.bfloat16, device=device)

    for _ in range(5):  # warmup
        dist.all_to_all_single(recv_buf, send_buf,
                               output_split_sizes=recv_counts,
                               input_split_sizes=send_counts)
        torch.xpu.synchronize(device)
    dist.barrier()

    times = []
    for _ in range(ITERS):
        t0 = time.perf_counter()
        dist.all_to_all_single(recv_buf, send_buf,
                               output_split_sizes=recv_counts,
                               input_split_sizes=send_counts)
        torch.xpu.synchronize(device)
        times.append((time.perf_counter() - t0) * 1e3)
    dist.barrier()

    times.sort()
    if rank == 0:
        max_load = max(sum(all_rank_counts[r]) for r in range(world_size))
        min_load = min(sum(all_rank_counts[r]) for r in range(world_size))
        imbalance = max_load / max(min_load, 1)
        p50 = times[len(times)//2]
        p95 = times[int(len(times)*0.95)]
        print(f"{label:<30}  imbalance={imbalance:.1f}x  p50={p50:.2f}ms  p95={p95:.2f}ms")

dist.destroy_process_group()

In [ ]:
result = subprocess.run(
    "mpirun -n 4 -ppn 4 python /tmp/ccl_moe_imbalance.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

## Summary

| Takeaway | Detail |
|---|---|
| **Alltoall = distributed transpose** | Rank i sends distinct data to each rank j; receives distinct data from each rank j |
| **MoE always uses alltoallv** | Token routing is imbalanced; fixed-size alltoall wastes bandwidth on padding |
| **Two alltoallv per MoE layer** | Dispatch (tokens → experts) and Combine (outputs → origin) |
| **Exchange send counts before dispatch** | Use alltoall on counts first so each rank knows recv buffer size |
| **Invariant:** `send_counts[j] == peer_j.recv_counts[me]` | Violation causes wrong data or hang — always verify in tests |
| **Load imbalance is a p95 killer** | Severe imbalance can 2–5× the alltoallv latency; measure in production |
| **topo algorithm on CRI** | oneCCL uses topo (scale-up + scatter scaleout) for alltoall; do not override |

**Next:** [End-to-End TP Decode Loop](04_inference_tp_decode.ipynb)